In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [24]:
# get_current_datetime tool function
from anthropic.types import ToolParam


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

## Tool Dispatch

Route a tool call by name to its Python implementation, and handle the `batch_tool` meta-tool.

In [ ]:
import json


def handle_batch_tool(invocations):
    """Execute multiple tool calls at once (batch_tool meta-tool).
    Each invocation: {"name": "...", "arguments": "<json string>"}
    Returns a list of individual results.
    """
    results = []
    for inv in invocations:
        name = inv.get("name")
        try:
            args = json.loads(inv.get("arguments", "{}"))
        except json.JSONDecodeError as e:
            results.append({"tool": name, "error": f"Invalid JSON arguments: {e}"})
            continue
        result = dispatch_tool(name, args)
        results.append({"tool": name, "result": result})
    return results


def dispatch_tool(name, args):
    """Route a tool call by name to its implementation."""
    if name == "get_current_datetime":
        return get_current_datetime(
            date_format=args.get("date_format", "%Y-%m-%d %H:%M:%S"),
        )
    elif name == "add_duration_to_datetime":
        return add_duration_to_datetime(
            datetime_str=args["datetime_str"],
            duration=args.get("duration", 0),
            unit=args.get("unit", "days"),
            input_format=args.get("input_format", "%Y-%m-%d"),
        )
    elif name == "set_reminder":
        return set_reminder(
            content=args["content"],
            timestamp=args["timestamp"],
        )
    elif name == "batch_tool":
        return handle_batch_tool(args.get("invocations", []))
    else:
        raise ValueError(f"Unknown tool: {name}")


# Quick sanity check
print(dispatch_tool("get_current_datetime", {}))
print(dispatch_tool("add_duration_to_datetime", {"datetime_str": "2025-01-01", "duration": 90, "unit": "days"}))

## Agentic Tool-Use Loop

The loop:
1. Call Claude with the current message history and tool schemas.
2. Collect any `tool_use` blocks from the response.
3. Append Claude's response turn to history.
4. Execute each tool call and build `tool_result` content blocks.
5. Append the tool results as a `user` turn.
6. Repeat until `stop_reason == 'end_turn'` (no more tool calls).

In [ ]:
ALL_TOOLS = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    batch_tool_schema,
]


def run_agent_loop(user_prompt, system=None):
    """Run a full agentic tool-use loop for a single user prompt.

    Returns the final text response from Claude.
    """
    messages = [{"role": "user", "content": user_prompt}]

    while True:
        response = chat(messages, system=system, tools=ALL_TOOLS)
        stop_reason = response.stop_reason
        print(f"[loop] stop_reason={stop_reason}  blocks={len(response.content)}")

        # Collect tool_use blocks
        tool_uses = []
        for block in response.content:
            if block.type == "text":
                print(f"  text: {block.text[:120]!r}")
            elif block.type == "tool_use":
                print(f"  tool_use: {block.name}({json.dumps(block.input)[:80]})")
                tool_uses.append(block)

        # Append Claude's full response turn to history
        add_assistant_message(messages, response)

        # Done when no tool calls remain
        if stop_reason == "end_turn" or not tool_uses:
            break

        # Execute tool calls and build tool_result content blocks
        tool_results = []
        for tu in tool_uses:
            try:
                result = dispatch_tool(tu.name, tu.input)
                result_str = json.dumps(result) if not isinstance(result, str) else result
                print(f"  result [{tu.name}]: {result_str[:120]!r}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tu.id,
                    "content": result_str,
                })
            except Exception as e:
                err_str = str(e)
                print(f"  error [{tu.name}]: {err_str}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tu.id,
                    "content": err_str,
                    "is_error": True,
                })

        # Feed results back as next user turn
        messages.append({"role": "user", "content": tool_results})

    # Return the final text from the last assistant turn
    return text_from_message(response)

## Demo — Single Tool Call

Ask Claude what today's date is. It should call `get_current_datetime`, receive the result, and answer.

In [ ]:
answer = run_agent_loop("What is today's date and time?")
print("\nFinal answer:", answer)

## Demo — Multi-Step Tool Calls

Ask Claude to compute a future date and then set a reminder for that date. This requires at least two tool calls: `add_duration_to_datetime` followed by `set_reminder`.

In [ ]:
answer = run_agent_loop(
    "What date is 90 days from 2025-01-01? "
    "Also set a reminder for that date with the message 'Q1 review deadline'."
)
print("\nFinal answer:", answer)

## Demo — batch_tool (parallel calls)

Ask Claude to do two date calculations at the same time using `batch_tool`.

In [ ]:
answer = run_agent_loop(
    "What day of the week was 2 weeks before 2025-06-20? "
    "Also, what date is 3 months after 2025-06-20? "
    "Answer both questions using a single batch tool call if possible."
)
print("\nFinal answer:", answer)